# Stock Price Prediction

### Libraries 

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import OneHotEncoder

#### Importing data

In [2]:
companies = ['Amazon', 'Apple', 'Google', 'Microsoft', 'Netflix']
data_dir = '../Data/stock_data/'

data = {}
for name in companies:
    file_path = os.path.join(data_dir, f"{name}.csv")
    df = pd.read_csv(file_path, parse_dates= ['Date'])
    df = df.sort_values('Date').reset_index(drop= True)
    df = df.rename(columns = {"Adj Close" : "Adj_Close"})
    df['Company'] = name
    data[name] = df

### EDA & Feature Engineering 

#### Feature Engineering

In [3]:
# daily return column
for name, df in data.items():
    df['Return'] = df['Adj_Close'].pct_change()
    data[name] = df

# label column 
for name, df in data.items():
    df['Label'] = (df['Adj_Close'].shift(-1) > df['Adj_Close']).astype(int)
    data[name] = df

# rolling moving average of 5/ 10/ 20 - days columns
for name, df in data.items():
    df['MA_5'] = df['Adj_Close'].rolling(window = 5).mean()
    df['MA_10'] = df['Adj_Close'].rolling(window = 10).mean()
    df['MA_20'] = df['Adj_Close'].rolling(window = 20).mean()
    data[name] = df

# rsi column
for name, df in data.items():
    change = df['Adj_Close'] - df['Adj_Close'].shift(1)
    df['gain'] = change.where(change > 0, 0)
    df['loss'] = -change.where(change < 0, 0)
    avg_gain = df['gain'].rolling(window = 14).mean()
    avg_loss = df['loss'].rolling(window = 14).mean()
    rs = avg_gain / avg_loss
    df['rsi'] = 100 - (100 / (1 + rs))
    data[name] = df

# rolling volatility
for name, df in data.items():
    df['Volatility'] = df['Return'].rolling(10).std()
    df['Volatility'] = df['Return'].rolling(20).std()
    data[name] = df

# volume change % 
for name, df in data.items():
    df['Vol_change'] = df['Volume'].pct_change()
    data[name] = df

#### Splitting the data

In [4]:
for name, df in data.items():
    print(name, df['Date'].min(), df['Date'].max())

Amazon 2000-01-03 00:00:00 2023-03-17 00:00:00
Apple 2000-01-03 00:00:00 2023-03-17 00:00:00
Google 2004-08-19 00:00:00 2023-03-17 00:00:00
Microsoft 2000-01-03 00:00:00 2023-03-17 00:00:00
Netflix 2002-05-23 00:00:00 2023-03-17 00:00:00


In [5]:
train_data = {}
test_data = {}
val_data = {}

for name, df in data.items():
    train_data[name] = df[df['Date'] <= '2018-12-31']
    val_data[name] = df[(df['Date'] >= '2019-01-01') & (df['Date'] <= '2021-06-30')]
    test_data[name] = df[df['Date'] >= '2021-07-01']

#### Dealing with null Values and combining data together

#### Choosing features to insert in the model

In [6]:
drop_col = ['Open', 'Close', 'High', 'Low', 'gain', 'loss']